## **Base de datos del proyecto**

Objetivo: almacenar en una base SQLite el dataset preparado para modelado, los resultados del modelo final y la configuración utilizada.

Esta etapa permite cumplir el requerimiento de la consigna relacionado con el almacenamiento de datos de entrada, predicciones, métricas y parametrización del modelo.

In [31]:
import pandas as pd
import sqlite3
from pathlib import Path

In [32]:
# Rutas del proyecto

ruta_dataset_modelado = Path("../data_processed/siniestros_viales_modelado.csv")
ruta_base_datos = Path("../database/siniestros_viales.db")

print("Dataset:", ruta_dataset_modelado)
print("Base de datos:", ruta_base_datos)

Dataset: ..\data_processed\siniestros_viales_modelado.csv
Base de datos: ..\database\siniestros_viales.db


In [33]:
# Carga del dataset

df = pd.read_csv(
    ruta_dataset_modelado,
    dtype={
        "rango_horario": str,
        "comuna_siniestro": str
    },
    low_memory=False
)

print(f"Registros: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

Registros: 65,818
Columnas: 17


In [34]:
# Vista general del dataset para verificar que se haya cargado correctamente

display(df.head())

,id_siniestro,siniestro_fatal,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,dia_semana,rango_horario,comuna_siniestro,tipo_de_via_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,latitud_siniestro,longitud_siniestro,edad_promedio_victimas,sexo_victima_agregado
0,LC-2019-0000053,0,2019-01-01,2019,1,1,Tuesday,1,8,AUTOPISTA,AUTO-UTILITARIO,AUTO,UTILITARIO,-34.669125,-58.443510,57.000000,M
1,LC-2019-0000647,0,2019-01-01,2019,1,1,Tuesday,18,12,AVENIDA,MOTO-AUTO,MOTO,AUTO,-34.583403,-58.490436,54.000000,M
2,LC-2019-0000445,0,2019-01-01,2019,1,1,Tuesday,13,11,NaN,AUTO-AUTO,AUTO,AUTO,-34.603255,-58.524662,40.333333,MIXTO
3,LC-2019-0000194,0,2019-01-01,2019,1,1,Tuesday,7,9,NaN,AUTO-CAMION,AUTO,CAMION,-34.650156,-58.528413,33.000000,F
4,LC-2019-0000329,0,2019-01-01,2019,1,1,Tuesday,12,4,NaN,AUTO-MOVIL,AUTO,MOVIL,-34.648387,-58.404748,23.000000,M


In [35]:
# Conexión a la base de datos SQLite

conexion = sqlite3.connect(ruta_base_datos)

print("Conexión creada correctamente.")

Conexión creada correctamente.


In [36]:
# Carga del dataset preparado en SQLite

df.to_sql(
    "siniestros_modelado",
    conexion,
    if_exists="replace",
    index=False
)

print("Tabla 'siniestros_modelado' creada correctamente.")

Tabla 'siniestros_modelado' creada correctamente.


In [37]:
# Verificación de la carga

consulta = """
SELECT COUNT(*) AS cantidad_registros
FROM siniestros_modelado
"""

pd.read_sql_query(consulta, conexion)

,cantidad_registros
0,65818


In [38]:
# Rutas de los resultados generados durante las fases de modelado

ruta_resultados_fase4 = Path("../data_processed/resultados_modelos_fase4.csv")
ruta_resultados_fase4b = Path("../data_processed/resultados_modelos_fase4b.csv")
ruta_predicciones_final = Path("../data_processed/predicciones_modelo_final.csv")

print("Resultados Fase 4:", ruta_resultados_fase4)
print("Resultados Fase 4B:", ruta_resultados_fase4b)
print("Predicciones modelo final:", ruta_predicciones_final)

Resultados Fase 4: ..\data_processed\resultados_modelos_fase4.csv
Resultados Fase 4B: ..\data_processed\resultados_modelos_fase4b.csv
Predicciones modelo final: ..\data_processed\predicciones_modelo_final.csv


In [39]:
# Carga de resultados de las fases de modelado

resultados_fase4 = pd.read_csv(ruta_resultados_fase4)
resultados_fase4b = pd.read_csv(ruta_resultados_fase4b)

print("Fase 4:", resultados_fase4.shape)
print("Fase 4B:", resultados_fase4b.shape)

Fase 4: (4, 9)
Fase 4B: (14, 9)


In [40]:
# Revisión de columnas disponibles en cada archivo

print("Columnas Fase 4:")
print(resultados_fase4.columns.tolist())

print("\nColumnas Fase 4B:")
print(resultados_fase4b.columns.tolist())

Columnas Fase 4:
['fase', 'experimento', 'modelo', 'semilla', 'accuracy', 'precision', 'recall', 'f1_score', 'es_modelo_final']

Columnas Fase 4B:
['fase', 'experimento', 'modelo', 'semilla', 'accuracy', 'precision', 'recall', 'f1_score', 'es_modelo_final']


In [41]:
# Consolidación de resultados de modelado

resultados_modelos = pd.concat(
    [resultados_fase4, resultados_fase4b],
    ignore_index=True
)

print(f"Registros consolidados: {resultados_modelos.shape[0]}")
print(f"Columnas consolidadas: {resultados_modelos.shape[1]}")

Registros consolidados: 18
Columnas consolidadas: 9


In [42]:
# Vista general de los resultados consolidados

display(resultados_modelos)

,fase,experimento,modelo,semilla,accuracy,precision,recall,f1_score,es_modelo_final
0,Fase 4,Modelado inicial,Dummy Classifier,42,0.989517,0.000000,0.000000,0.000000,0
1,Fase 4,Modelado inicial,Logistic Regression,42,0.867669,0.057883,0.760870,0.107582,0
2,Fase 4,Modelado inicial,Decision Tree,42,0.979793,0.104938,0.123188,0.113333,0
3,Fase 4,Modelado inicial,Random Forest,42,0.989517,0.000000,0.000000,0.000000,0
4,Fase 4B,Modelos avanzados,Balanced RF,42,0.886964,0.067862,0.768116,0.124706,0
5,Fase 4B,Modelos avanzados,SMOTE + Random Forest,42,0.989289,0.400000,0.043478,0.078431,0
6,Fase 4B,Modelos avanzados,XGBoost,42,0.937101,0.095070,0.586957,0.163636,0
7,Fase 4B,Feature Engineering temporal,Random Forest + Feature Engineering temporal,42,0.973412,0.139456,0.297101,0.189815,1
8,Fase 4B,Validación multisemilla,Random Forest oficial,42,NaN,NaN,NaN,0.170100,0
9,Fase 4B,Validación multisemilla,Random Forest oficial,7,NaN,NaN,NaN,0.243400,0


In [44]:
# Persistencia de resultados de modelado en SQLite

resultados_modelos.to_sql(
    "resultados_modelos",
    conexion,
    if_exists="replace",
    index=False
)

print("Tabla 'resultados_modelos' creada correctamente.")

Tabla 'resultados_modelos' creada correctamente.


In [45]:
# Listado de tablas existentes en SQLite

consulta = """
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
"""

pd.read_sql_query(consulta, conexion)

,name
0,resultados_modelos
1,siniestros_modelado


In [46]:
# Verificación de la tabla resultados_modelos

consulta = """
SELECT COUNT(*) AS cantidad_registros
FROM resultados_modelos
"""

pd.read_sql_query(consulta, conexion)

,cantidad_registros
0,18


In [47]:
# Carga de predicciones del modelo final

predicciones_modelo_final = pd.read_csv(ruta_predicciones_final)

print(f"Registros: {predicciones_modelo_final.shape[0]:,}")
print(f"Columnas: {predicciones_modelo_final.shape[1]}")

Registros: 13,164
Columnas: 4


In [48]:
# Vista general de las predicciones

display(predicciones_modelo_final.head())

,id_siniestro,valor_real,valor_predicho,probabilidad_fatal
0,LC-2022-0517979,0,0,0.007911
1,LC-2019-0496495,0,0,0.045694
2,LC-2024-0010569,0,0,0.230478
3,LC-2024-0147333,0,0,0.002914
4,LC-2019-0573464,0,0,0.058202


In [49]:
# Persistencia de predicciones del modelo final en SQLite

predicciones_modelo_final.to_sql(
    "predicciones_modelo_final",
    conexion,
    if_exists="replace",
    index=False
)

print("Tabla 'predicciones_modelo_final' creada correctamente.")

Tabla 'predicciones_modelo_final' creada correctamente.


In [50]:
# Listado de tablas existentes en SQLite

consulta = """
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
"""

pd.read_sql_query(consulta, conexion)

,name
0,predicciones_modelo_final
1,resultados_modelos
2,siniestros_modelado


In [51]:
# Resumen de la base de datos

for tabla in [
    "siniestros_modelado",
    "resultados_modelos",
    "predicciones_modelo_final"
]:
    
    consulta = f"""
    SELECT COUNT(*) AS cantidad_registros
    FROM {tabla}
    """
    
    cantidad = pd.read_sql_query(
        consulta,
        conexion
    ).iloc[0, 0]
    
    print(f"{tabla}: {cantidad:,} registros")

siniestros_modelado: 65,818 registros
resultados_modelos: 18 registros
predicciones_modelo_final: 13,164 registros


In [52]:
# Cierre de la conexión a SQLite

conexion.close()

print("Conexión cerrada correctamente.")

Conexión cerrada correctamente.
